# Hausa→English speech-to-text translation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsuxalo/Spoken-Language-Translation-Model/blob/main/capstone_demo.ipynb)

This graduate-project notebook drives the reusable `hausa_s2tt` package. It does not embed results or reimplement the pipeline. Cells are marked **FAST DEMO** or **EXPENSIVE TRAINING**. The repository is partial until a full direct model and the one-time held-out comparison exist.

## 1. Project objective

Compare zero-shot Whisper, Hausa ASR, an ASR→NLLB cascade, and a genuinely English-supervised direct Whisper model. ASR produces Hausa; S2TT produces English.

In [ ]:
SYSTEMS = {
    'zero_shot': 'Hausa audio → base Whisper translate → English',
    'asr': 'Hausa audio → Hausa-fine-tuned Whisper → Hausa',
    'cascade': 'Hausa audio → Hausa ASR → NLLB hau_Latn→eng_Latn → English',
    'direct': 'Hausa audio → English-supervised Whisper → English',
}
SYSTEMS

## 2. Architecture

Zero-shot and fine-tuned direct systems share the Whisper architecture but not the training claim. The cascade exposes an intermediate Hausa transcript. All Whisper paths use validated mono 16 kHz audio and at-most-29-second chunks.

## 3. Environment setup — FAST DEMO

Run this cell once in a fresh Colab session. `SOURCE_MODE='archive'` is reproducible before this local branch is published: create the ZIP locally with `git archive --format=zip --prefix=Spoken-Language-Translation-Model/ HEAD -o hausa-s2tt-source.zip`, then upload it when prompted. Use Git mode only after the explicit `REPO_REF` exists remotely.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/tsuxalo/Spoken-Language-Translation-Model.git'
REPO_REF = 'feature/direct-s2tt'  # Must exist remotely when SOURCE_MODE='git'.
SOURCE_MODE = 'archive'  # 'archive' for unpublished work; 'git' after push/merge.
REPO_DIR = Path('/content/Spoken-Language-Translation-Model')
if 'google.colab' in sys.modules:
    if not REPO_DIR.exists():
        if SOURCE_MODE == 'git':
            subprocess.run(
                ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
                check=True,
            )
        elif SOURCE_MODE == 'archive':
            from google.colab import files
            uploaded = files.upload()
            archives = [name for name in uploaded if name.lower().endswith('.zip')]
            if len(archives) != 1:
                raise RuntimeError('Upload exactly one source ZIP created with the documented git archive command.')
            shutil.unpack_archive(str(Path('/content') / archives[0]), '/content')
        else:
            raise ValueError("SOURCE_MODE must be 'archive' or 'git'")
    if not (REPO_DIR / 'pyproject.toml').is_file():
        raise RuntimeError(f'Expected a project checkout at {REPO_DIR}')
    os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('workspace:', Path.cwd())

## 4. Hardware detection — FAST DEMO

Do not assume Colab supplies a particular accelerator. The package selects BF16, FP16, or FP32 from the detected runtime.

In [ ]:
from hausa_s2tt.hardware import hardware_snapshot, select_precision

print(hardware_snapshot())
print(select_precision().to_dict())

## 5. Dataset exploration — FAST DEMO

The tracked report is safe to inspect. A fresh complete metadata audit takes a few minutes and excludes audio bytes.

In [ ]:
import json
from pathlib import Path

audit = json.loads(Path('reports/naija_s2st_audit_summary.json').read_text(encoding='utf-8'))
print(json.dumps(audit, indent=2))
# Reproduce when desired:
# !hausa-s2tt-data naija --source parquet --workers 4 --output-dir artifacts/audits/naija_s2st

## 6. Hausa–English pair verification — FAST DEMO

NaijaS2ST uses language-prefixed IDs. We retain original IDs and align only after removing the prefix that matches the row language.

In [ ]:
from hausa_s2tt.datasets import alignment_key

assert alignment_key('ETE_0001', 'english') == 'TE_0001'
assert alignment_key('HTE_0001', 'hausa') == 'TE_0001'
print('Verified language-prefix alignment semantics; direct labels are English target_text.')

## 7. ASR baseline — FAST DEMO

Upload or mount a Hausa WAV file and set `AUDIO_PATH`. This path returns Hausa, not English.

In [ ]:
AUDIO_PATH = None  # e.g. '/content/sample.wav'
if AUDIO_PATH:
    from hausa_s2tt.inference import create_asr_runtime
    from hausa_s2tt.revisions import HAUSA_ASR_ID, HAUSA_ASR_REVISION
    asr = create_asr_runtime(HAUSA_ASR_ID, revision=HAUSA_ASR_REVISION)
    print(asr.process(AUDIO_PATH).to_dict())
else:
    print('Set AUDIO_PATH to run Hausa ASR.')

## 8. Direct zero-shot translation baseline — FAST DEMO

Base Whisper with `task=translate` emits English without project fine-tuning. It is the zero-shot baseline.

In [ ]:
if AUDIO_PATH:
    from hausa_s2tt.inference import create_zero_shot_runtime
    from hausa_s2tt.revisions import WHISPER_SMALL_ID, WHISPER_SMALL_REVISION
    zero_shot = create_zero_shot_runtime(WHISPER_SMALL_ID, revision=WHISPER_SMALL_REVISION)
    print(zero_shot.process(AUDIO_PATH).to_dict())
else:
    print('Set AUDIO_PATH to run zero-shot translation.')

## 9. Direct S2TT training — EXPENSIVE TRAINING

This downloads the NaijaS2ST training audio and optimizes against genuine English labels. Run a hardware-matched pilot first and obtain authorization if the full job is projected beyond the resource guardrails. Official dev is not loaded by the trainer.

In [ ]:
RUN_EXPENSIVE_TRAINING = False
training_command = ['hausa-s2tt-train', '--config', 'configs/direct_s2tt_full.yaml']
print(' '.join(training_command))
if RUN_EXPENSIVE_TRAINING:
    subprocess.run(training_command, check=True)
else:
    print('Skipped. Run hausa-s2tt-smoke-train first.')

## 10. Cascade translation — FAST DEMO

The cascade exposes both the Hausa ASR transcript and the English NLLB output. NLLB uses `hau_Latn` → `eng_Latn` and has a non-commercial license.

In [ ]:
if AUDIO_PATH:
    from hausa_s2tt.cascade import CascadeTranslator
    from hausa_s2tt.inference import create_asr_runtime
    from hausa_s2tt.mt import NLLBTranslator
    from hausa_s2tt.revisions import HAUSA_ASR_REVISION, NLLB_REVISION
    cascade = CascadeTranslator(
        asr=create_asr_runtime(revision=HAUSA_ASR_REVISION),
        mt=NLLBTranslator(revision=NLLB_REVISION),
    )
    print(cascade.translate(AUDIO_PATH).to_dict())
else:
    print('Set AUDIO_PATH to run the cascade.')

## 11. Metrics — FAST DEMO

ASR uses corpus raw/normalized WER and CER. Translation uses SacreBLEU plus chrF++ with signatures.

In [ ]:
from hausa_s2tt.metrics import compute_asr_metrics, compute_translation_metrics

print(compute_asr_metrics(['Sannu, duniya!'], ['sannu duniya']))
print(compute_translation_metrics(['Good morning.'], ['Good morning.']))

## 12. Efficiency measurements — FAST DEMO

Run the three-step smoke on the current hardware. It records wall time, throughput, RTF, GPU-hours, and peak VRAM when CUDA is present. Tiny-smoke timing is not a Whisper-small estimate.

In [ ]:
RUN_TRAINING_SMOKE = False
if RUN_TRAINING_SMOKE:
    subprocess.run(['hausa-s2tt-smoke-train', '--output-dir', 'artifacts/smoke'], check=True)
else:
    print('Set RUN_TRAINING_SMOKE=True for three CPU/GPU-safe steps.')

## 13. Side-by-side examples — FAST DEMO

Final evaluation artifacts contain identical source IDs for each system. Join them only after the guarded final run.

In [ ]:
import pandas as pd

comparison_root = Path('artifacts/evaluations/s2tt/final-v1')
if comparison_root.exists():
    comparison = None
    for system in ['zero_shot', 'direct', 'cascade']:
        path = comparison_root / system / 'predictions.jsonl'
        if path.exists():
            frame = pd.read_json(path, lines=True)[['source_id', 'reference', 'prediction']]
            frame = frame.rename(columns={'prediction': system})
            if comparison is None:
                comparison = frame
            else:
                comparison = comparison.merge(frame[['source_id', system]], on='source_id', validate='one_to_one')
    display(comparison.head(10) if comparison is not None else pd.DataFrame())
else:
    print('No final comparison artifact exists; no example is fabricated.')

## 14. Error analysis — FAST DEMO

Review names, numbers, dates, long/noisy audio, negation, omissions, additions, dialect variation, and ASR propagation.

In [ ]:
from hausa_s2tt.evaluation import analysis_flags

analysis_flags('An fara taro ranar 12 Mayu.', 'The meeting began on 12 May.', 21.0)

## 15. Final comparison — EXPENSIVE TRAINING / ONE-TIME FINAL EVALUATION

Freeze the direct checkpoint and generation settings first. The command requires explicit confirmation and seals the run name to prevent casual repeated final-set inspection.

In [ ]:
RUN_FINAL_EVALUATION = False
DIRECT_CHECKPOINT = 'artifacts/checkpoints/whisper-small-ha-en-s2tt'
final_command = [
    'hausa-s2tt-evaluate', '--systems', 'zero_shot', 'direct', 'cascade',
    '--direct-model-id', DIRECT_CHECKPOINT, '--run-name', 'final-v1',
    '--dataset-revision', '898f51582750fe244693794f22e3f4b32c5baf95',
    '--confirm-final-test',
]
print(' '.join(final_command))
if RUN_FINAL_EVALUATION:
    subprocess.run(final_command, check=True)
else:
    print('Skipped: model selection is not complete.')

## 16. Limitations

No full direct model or clean final comparison exists yet. The full 69.9 GB NaijaS2ST audio was not decoded for corruption. Dialect, regional, accent, speaker, gender, topic, and recording-condition coverage remains incomplete. Direct and cascade outputs can omit, mistranslate, or hallucinate content. NLLB is CC BY-NC 4.0. This prototype is not a sole source for consequential decisions.

## 17. Future work

The next experiment is a matched, hardware-measured LoRA pilot comparing base-Whisper and Hausa-ASR initialization on the same speaker-safe validation subset. After selection: run one final comparison, populate model cards from artifacts, and conduct blinded fluent-speaker evaluation.